# ResNet-18 Unfreeze + Penalty Regularization

## Motivation
The unfrozen ResNet baseline showed heavy overfitting — train F2 reached ~0.97 while val F2 plateaued ~0.74. This notebook adds explicit L1/L2 weight penalty terms directly to the loss to constrain weight magnitudes and reduce overfitting.

## Why explicit penalty vs `weight_decay` in Adam?
Adam's `weight_decay` parameter is **not** equivalent to L2 penalty. Adam scales updates by per-parameter gradient magnitude, so `weight_decay` acts more like a scaled shrinkage rather than a true L2 penalty. Adding the penalty directly to the loss gives the correct, unscaled regularization effect.

## Lambda Sweep
| Run | l1_lambda | l2_lambda | Purpose |
|-----|-----------|-----------|--------|
| 0   | 0         | 0         | Baseline (no penalty) |
| 1   | 0         | 1e-4      | Mild L2 |
| 2   | 0         | 1e-3      | Moderate L2 |
| 3   | 0         | 1e-2      | Strong L2 |
| 4   | 1e-4      | 0         | Mild L1 |

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import fbeta_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

from src.data.dataloader import get_dataloaders
from src.models.resnet import get_resnet
from src.training.trainer import train_one_epoch, validate_one_epoch

In [ ]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

set_seed(42)

In [ ]:
use_mps = True
print('MPS available:', torch.backends.mps.is_available())
device = torch.device('mps' if (use_mps and torch.backends.mps.is_available()) else 'cpu')
print(f'Using device: {device}')

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    train_csv='../../data/splits/train.csv',
    val_csv='../../data/splits/val.csv',
    test_csv='../../data/splits/test.csv',
    image_dir='../../data/raw/HAM10000/images',
    batch_size=32,
    image_size=224,
    num_workers=0,
)

train_df = pd.read_csv('../../data/splits/train.csv')

num_melanoma = (train_df['label'] == 0).sum()
num_nevus = (train_df['label'] == 1).sum()

pos_weight = torch.tensor([num_nevus / num_melanoma], dtype=torch.float32).to(device)
print('Positive weight:', pos_weight)

## Lambda Sweep

In [ ]:
runs = [
    {'label': 'Baseline',    'l1_lambda': 0.0,  'l2_lambda': 0.0},
    {'label': 'L2=1e-4',     'l1_lambda': 0.0,  'l2_lambda': 1e-4},
    {'label': 'L2=1e-3',     'l1_lambda': 0.0,  'l2_lambda': 1e-3},
    {'label': 'L2=1e-2',     'l1_lambda': 0.0,  'l2_lambda': 1e-2},
    {'label': 'L1=1e-4',     'l1_lambda': 1e-4, 'l2_lambda': 0.0},
]

num_epochs = 20
all_results = {}

for run in runs:
    run_label = run['label']
    l1_lambda = run['l1_lambda']
    l2_lambda = run['l2_lambda']

    print(f'\n=== Run: {run_label} (l1={l1_lambda}, l2={l2_lambda}) ===')
    set_seed(42)

    model = get_resnet(num_classes=1, freeze_backbone=False).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    history = {'train_loss': [], 'val_loss': [], 'train_f2': [], 'val_f2': []}
    best_val_f2 = 0.0
    best_state = None

    for epoch in range(num_epochs):
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            l1_lambda=l1_lambda, l2_lambda=l2_lambda
        )
        val_metrics = validate_one_epoch(model, val_loader, criterion, device)

        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['train_f2'].append(train_metrics['f2'])
        history['val_f2'].append(val_metrics['f2'])

        print(
            f"  Epoch [{epoch+1}/{num_epochs}] | "
            f"Train Loss: {train_metrics['loss']:.4f}, F2: {train_metrics['f2']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f}, F2: {val_metrics['f2']:.4f}"
        )

        if val_metrics['f2'] > best_val_f2:
            best_val_f2 = val_metrics['f2']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'    -> New best val F2: {best_val_f2:.4f}')

    all_results[run_label] = {
        'history': history,
        'best_val_f2': best_val_f2,
        'best_state': best_state,
    }
    print(f'Best val F2 for {run_label}: {best_val_f2:.4f}')

## Compare Overfitting: Train vs Val F2 Gap

In [ ]:
print(f"{'Run':<12} | {'Best Val F2':>11} | {'Final Train F2':>14} | {'Final Val F2':>12} | {'Gap (T-V)':>9}")
print('-' * 70)
for run_label, result in all_results.items():
    h = result['history']
    final_train_f2 = h['train_f2'][-1]
    final_val_f2 = h['val_f2'][-1]
    gap = final_train_f2 - final_val_f2
    print(f"{run_label:<12} | {result['best_val_f2']:>11.4f} | {final_train_f2:>14.4f} | {final_val_f2:>12.4f} | {gap:>9.4f}")

## Training Curves

In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for run_label, result in all_results.items():
    h = result['history']
    axes[0].plot(epochs, h['train_f2'], label=f'{run_label} train', linestyle='--')
    axes[0].plot(epochs, h['val_f2'], label=f'{run_label} val')

axes[0].set_title('F2 Score: Train vs Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F2')
axes[0].legend(fontsize=7)

for run_label, result in all_results.items():
    h = result['history']
    axes[1].plot(epochs, h['train_loss'], label=f'{run_label} train', linestyle='--')
    axes[1].plot(epochs, h['val_loss'], label=f'{run_label} val')

axes[1].set_title('Loss: Train vs Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## Save & Evaluate Best Run

In [ ]:
best_run_label = max(all_results, key=lambda k: all_results[k]['best_val_f2'])
print(f'Best run: {best_run_label} (val F2 = {all_results[best_run_label]["best_val_f2"]:.4f})')

best_model = get_resnet(num_classes=1, freeze_backbone=False).to(device)
best_model.load_state_dict({k: v.to(device) for k, v in all_results[best_run_label]['best_state'].items()})

torch.save(best_model.state_dict(), '../../models/resnet_unfreeze_penalty_best.pth')
print('Saved to models/resnet_unfreeze_penalty_best.pth')

## Threshold Tuning

In [ ]:
best_model.eval()

val_probs = []
val_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = best_model(images.to(device))
        probs = torch.sigmoid(outputs).squeeze(1)
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(labels.numpy())

thresholds = np.arange(0.01, 0.9, 0.01)
f2_scores = [
    fbeta_score(val_labels, (np.array(val_probs) >= t).astype(int), beta=2, pos_label=1, zero_division=0)
    for t in thresholds
]

best_threshold = thresholds[np.argmax(f2_scores)]
print(f'Best threshold: {best_threshold:.2f} | Val F2: {max(f2_scores):.4f}')

## Test Set Evaluation

In [ ]:
best_model.eval()

all_labels = []
all_probs = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs >= best_threshold).long()

        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix:')
print(cm)
print()
print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Melanoma', 'Melanoma'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix — Best Run: {best_run_label}')
plt.show()